# Capítulo 11: Regressão Linear Simples

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 14 de Grus (2019).

> A arte, como a moralidade, consiste em traçar a linha em algum lugar.
>
> — G. K. Chesterton

Correlação mede se duas variáveis andam juntas — um único número, entre -1 e 1, que resume a força de uma relação linear. Mas "andam juntas" não é a mesma coisa que "eu sei prever uma a partir da outra". Este capítulo dá esse passo: em vez de um número que resume a força de uma relação, ajustamos os dois parâmetros — inclinação e intercepto — que descrevem a relação por completo. No fim dele, dado quantos amigos um usuário tem, você tem uma equação que produz um palpite concreto, em minutos, para quanto tempo ele passa no site.

É o primeiro capítulo deste livro em que o ajuste produz um **parâmetro interpretável, com unidade do mundo**. Os capítulos 1 a 4 construíram peças — vetores, distância, visualização. O [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) construiu a máquina de otimização. O [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) deu o vocabulário para julgar se um modelo é bom. O [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html) montou o primeiro classificador completo e o [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html), o primeiro modelo probabilístico — mas o que aqueles dois ajustam são contagens e probabilidades por palavra, grandezas internas ao algoritmo. Aqui, pela primeira vez, um parâmetro ajustado sai medido em unidade do problema: **0,9 minuto por amigo**. "Cada amigo a mais corresponde a quase um minuto a mais por dia no site" é uma frase que se leva para uma reunião, não só para um notebook.

Este capítulo também cobra duas dívidas. A primeira é do [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/03-hipotese-motivadora-datasciencester.html): ao fatiar salários em faixas arbitrárias de tempo de casa, aquele capítulo admitiu que o que se queria de verdade era uma afirmação sobre o efeito de **mais um ano** de experiência sobre o salário médio, e adiou para cá. É exatamente o que a inclinação de uma regressão é. A segunda é da [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html), que prometeu que a regressão linear simples tem uma **fórmula fechada** — duas médias, uma covariância, uma variância, sem laço nenhum — e adiou a demonstração para aqui. A seção 11.1 cumpre essa promessa; a seção 11.2 resolve o mesmo problema por gradiente descendente, o caminho construído no Capítulo 5; e as duas chegam ao mesmo lugar. A seção 11.3 explica por que esse "mesmo lugar" não é coincidência: por que minimizar o erro quadrático é, sob uma suposição razoável sobre os erros, exatamente a coisa certa a fazer — e não apenas uma escolha conveniente.

Ao final deste capítulo, você será capaz de:

- Escrever o modelo de regressão linear simples e nomear seus dois parâmetros
- Ajustar inclinação e intercepto por fórmula fechada, sem nenhum laço de otimização
- Ajustar os mesmos parâmetros por gradiente descendente e reconhecer que o resultado converge para o mesmo lugar que a fórmula fechada
- Calcular o coeficiente de determinação (R²) de um modelo ajustado e interpretar o que ele mede
- Explicar por que minimizar o erro quadrático equivale a maximizar a verossimilhança quando os erros são normais
- Demonstrar, com números na tela, por que se trabalha com a log-verossimilhança e não com a verossimilhança
- Reconhecer o efeito de um único outlier sobre uma correlação e sobre a reta ajustada
- Explicar por que a regressão linear simples tem fórmula fechada e outros modelos deste livro não têm

## Seções

| Seção | Tópico |
|---|---|
| [11.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) | O Modelo |
| [11.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/02-usando-gradiente-descendente.html) | Usando Gradiente Descendente |
| [11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html) | Estimação por Máxima Verossimilhança |

## O Modelo

> **📌 Nota**
>
> Esta seção corresponde a *The Model*, do capítulo 14 de Grus (2019).

Voltamos à rede social fictícia DataSciencester, que aparece em vários capítulos deste livro. A pergunta de negócio é simples de enunciar: existe relação entre quantos amigos um usuário tem e quanto tempo por dia ele passa no site? E se existe, dá para descrevê-la com números — não só "existe uma tendência", mas "cada amigo a mais vale, em média, tantos minutos a mais"?

O primeiro passo, que você provavelmente já viu antes de chegar a este livro, é medir a **correlação**: um único número entre -1 e 1 que resume o quanto duas variáveis variam juntas. `scratch/statistics.py` já traz essa função pronta, junto com os dados da própria DataSciencester:

In [ ]:
from scratch.statistics import (num_friends, daily_minutes,
                                num_friends_good, daily_minutes_good,
                                mean, correlation)
import matplotlib.pyplot as plt
plt.close('all')

São duas listas paralelas, uma entrada por usuário: `num_friends_good` guarda quantos amigos cada um tem, `daily_minutes_good` guarda quantos minutos por dia ele passa no site. O sufixo `_good` marca a versão **limpa** dos dados — o livro-texto descartou um usuário antes de calcular qualquer coisa, e daqui a pouco vamos ver qual e por quê.

In [ ]:
len(num_friends_good), mean(num_friends_good), mean(daily_minutes_good)

In [ ]:
correlation(num_friends_good, daily_minutes_good)

São 203 usuários. Em média, cada um tem quase 7 amigos e passa cerca de 29 minutos por dia no site. E a correlação — pouco mais de 0,57 — diz que existe uma tendência real: mais amigos, mais tempo. Mas repare no que esse número **não** diz. Ele não diz se um amigo a mais vale, em média, 10 segundos ou 10 minutos a mais de uso. Ele não dá uma equação. Para isso, precisamos de outra coisa: um modelo.

> **⚠️ Atenção — Um ponto em 204 derruba a correlação a 43%**
>
> As listas cruas — `num_friends` e `daily_minutes`, sem o sufixo — têm 204 usuários. O livro-texto descartou um deles: alguém que declarava **100 amigos** e passava **1 minuto por dia** no site. Cem amigos é mais que o dobro do segundo colocado (49), e um minuto por dia é o piso da amostra; a combinação das duas coisas é quase certamente uma conta de teste ou um erro de captura, não um usuário. Veja o que esse único ponto faz com a correlação:

In [ ]:
len(num_friends), correlation(num_friends, daily_minutes)

> Com ele dentro, 0,2474. Sem ele, 0,5737. **Um ponto em 204 derruba a correlação para 43% do valor** — porque ele fica no extremo direito do eixo `x`, onde cada ponto tem alavanca desproporcional, e aponta para baixo enquanto todo o resto aponta para cima.
>
> Isso é um aviso permanente, não uma curiosidade deste conjunto: a correlação — e, como o resto desta seção vai mostrar, a regressão que se calcula a partir dela — não tem defesa nenhuma contra um outlier. Nenhuma mensagem de erro avisa; o número simplesmente sai outro. E nenhum resumo numérico denuncia o problema: os 0,2474 são um número perfeitamente plausível de se publicar. Quem viu o ponto foi quem olhou a nuvem de pontos — que é para o que serve o gráfico de dispersão logo adiante nesta seção.

### O modelo

> **❗ Importante**
>
> Vamos supor que você já se convenceu de que ter mais amigos **causa** mais tempo no site — e não o contrário (quem já passa mais tempo no site tende a fazer mais amigos ali), nem um terceiro fator que produz os dois efeitos ao mesmo tempo (por exemplo, ter mais tempo livre em geral). Correlação sozinha nunca prova isso; é uma suposição que se traz de fora dos dados, não uma conclusão que se tira deles. O capítulo segue com essa suposição porque o objetivo aqui é regressão, não inferência causal — mas vale lembrar que ela está sendo feita.

Com essa suposição em mãos, supomos que existam constantes $\alpha$ (alfa) e $\beta$ (beta) tais que:

$$
y_i = \beta x_i + \alpha + \varepsilon_i
$$

em que $y_i$ é o número de minutos que o usuário $i$ passa no site por dia, $x_i$ é o número de amigos que o usuário $i$ tem, e $\varepsilon_i$ é um termo de erro — esperamos que pequeno — que representa outros fatores não capturados por este modelo simples: o que o usuário faz da vida, se está de férias, quantos outros aplicativos disputam a atenção dele, e assim por diante.

$\beta$ é a **inclinação**: quantos minutos a mais por dia um amigo adicional está associado a produzir. $\alpha$ é o **intercepto**: quantos minutos por dia o modelo prevê para um usuário com zero amigos.

### Prevendo e medindo o erro

Supondo que já sabemos escolher $\alpha$ e $\beta$, fazer previsões é imediato:

In [ ]:
def predict(alpha: float, beta: float, x_i: float) -> float:
    return beta * x_i + alpha

def error(alpha: float, beta: float, x_i: float, y_i: float) -> float:
    """
    O erro ao prever beta * x_i + alpha
    quando o valor real é y_i
    """
    return predict(alpha, beta, x_i) - y_i

Como escolher $\alpha$ e $\beta$? Qualquer escolha das duas nos dá uma previsão para cada `x_i`. Como conhecemos o valor real `y_i`, podemos calcular o erro de cada par. O que gostaríamos de saber é o erro **total** sobre o conjunto de dados inteiro — mas somar os erros direto não serve: se a previsão para `x_1` está alta demais e a previsão para `x_2` está baixa demais na mesma proporção, os erros se cancelam e escondem um modelo ruim. A saída é somar os erros **ao quadrado**:

In [ ]:
from scratch.linear_algebra import Vector

def sum_of_sqerrors(alpha: float, beta: float, x: Vector, y: Vector) -> float:
    return sum(error(alpha, beta, x_i, y_i) ** 2
               for x_i, y_i in zip(x, y))

A solução de **mínimos quadrados** é escolher o $\alpha$ e o $\beta$ que deixam `sum_of_sqerrors` a menor possível.

### A fórmula fechada

Aqui está a dívida deste capítulo com o [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html). Ao apresentar gradiente descendente, aquele capítulo mostrou como ajustar `theta = [inclinação, intercepto]` iterando — um passo pequeno de cada vez, na direção oposta ao gradiente do erro quadrático médio — e prometeu que a regressão linear simples tinha um jeito mais direto: uma **fórmula fechada**.

Usando cálculo (ou álgebra tediosa, nas palavras do próprio Grus), o $\alpha$ e o $\beta$ que minimizam o erro são dados por:

In [ ]:
from typing import Tuple
from scratch.linear_algebra import Vector
from scratch.statistics import correlation, standard_deviation, mean

def least_squares_fit(x: Vector, y: Vector) -> Tuple[float, float]:
    """
    Dados dois vetores x e y,
    encontra os valores de alpha e beta de mínimos quadrados
    """
    beta = correlation(x, y) * standard_deviation(y) / standard_deviation(x)
    alpha = mean(y) - beta * mean(x)
    return alpha, beta

> **🔷 Conceito**
>
> Nada de laço, nada de `theta` inicial aleatório, nada de taxa de aprendizado. `least_squares_fit` calcula duas médias, uma correlação (que por baixo é uma covariância dividida por dois desvios padrão) e dois desvios padrão — e devolve a resposta exata. É exatamente o que a [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html) prometeu: **duas médias, uma covariância, uma variância, sem laço nenhum**.
>
> Essa fórmula existe porque o problema é simples o bastante para ter solução fechada: derive `sum_of_sqerrors` em relação a $\alpha$ e a $\beta$, iguale as duas derivadas a zero, resolva o sistema — e o que sobra são só médias, variância e covariância. A [seção 11.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/02-usando-gradiente-descendente.html) resolve exatamente o mesmo problema pelo caminho do Capítulo 5, iterando em vez de calculando direto, e as duas respostas vão bater.

Sem passar pela conta exata, vale pensar no porquê disso fazer sentido. A escolha de $\alpha$ diz que, ao ver o valor **médio** da variável explicativa `x`, prevemos o valor **médio** do alvo `y` — é por isso que `alpha = mean(y) - beta * mean(x)`. A escolha de $\beta$ diz que, a cada aumento de um desvio padrão em `x`, a previsão aumenta `correlation(x, y)` desvios padrão de `y` — de modo que correlação 1 devolve o aumento inteiro, correlação -1 devolve uma queda de mesmo tamanho, e correlação 0 zera `beta`, deixando a previsão insensível a `x`.

Como sempre, escrevemos um teste rápido:

In [ ]:
x = [i for i in range(-100, 110, 10)]
y = [3 * i - 5 for i in x]

# Deveria encontrar que y = 3x - 5
assert least_squares_fit(x, y) == (-5, 3)

Agora é fácil aplicar isso aos dados sem outlier que já carregamos:

In [ ]:
alpha, beta = least_squares_fit(num_friends_good, daily_minutes_good)
assert 22.9 < alpha < 23.0
assert 0.9 < beta < 0.905

alpha, beta

Isso dá $\alpha \approx 22{,}95$ e $\beta \approx 0{,}904$. O modelo diz que esperamos que um usuário com $n$ amigos passe $22{,}95 + n \times 0{,}904$ minutos por dia no site. Ou seja: prevemos que um usuário sem nenhum amigo no DataSciencester ainda passaria uns 23 minutos por dia no site — e que cada amigo adicional está associado a quase mais um minuto de uso diário.

### Olhando o ajuste

Como o [Capítulo 3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap03/04-graficos-de-dispersao.html) mostrou, um gráfico de dispersão é a ferramenta certa para duas variáveis pareadas. Aquele capítulo já desenhou "amigos × minutos", mas com nove pontos inventados para ilustrar o tipo de gráfico, numa escala completamente outra (60 a 72 amigos, 105 a 220 minutos); aqui são os 203 usuários reais da DataSciencester. Vamos usar o mesmo `plt.scatter`, agora com esses pontos, e desenhar a reta ajustada por cima:

In [ ]:
# Figura: Modelo de regressão linear simples sobre os dados da DataSciencester
plt.scatter(num_friends_good, daily_minutes_good)

x_reta = [min(num_friends_good), max(num_friends_good)]
y_reta = [predict(alpha, beta, x_i) for x_i in x_reta]
plt.plot(x_reta, y_reta, color='black')

plt.title("Regressão linear simples")
plt.xlabel("# de amigos")
plt.ylabel("minutos por dia")
plt.show()

A reta passa razoavelmente perto da nuvem de pontos, mas está longe de tocar a maioria deles — a dispersão vertical em torno da reta é grande, principalmente para valores baixos de `x`, onde a maior parte dos dados está concentrada.

### Quão bom é o ajuste?

Olhar o gráfico dá uma impressão, mas "razoavelmente perto" não é uma métrica. A métrica usual para regressão é o **coeficiente de determinação** (ou R²), que mede a fração da variação total em `y` que o modelo captura:

In [ ]:
from scratch.statistics import de_mean

def total_sum_of_squares(y: Vector) -> float:
    """a variação quadrática total dos y_i em torno da média deles"""
    return sum(v ** 2 for v in de_mean(y))

def r_squared(alpha: float, beta: float, x: Vector, y: Vector) -> float:
    """
    a fração da variação em y capturada pelo modelo, que é igual a
    1 menos a fração da variação em y NÃO capturada pelo modelo
    """
    return 1.0 - (sum_of_sqerrors(alpha, beta, x, y) /
                  total_sum_of_squares(y))

rsq = r_squared(alpha, beta, num_friends_good, daily_minutes_good)
assert 0.328 < rsq < 0.330

rsq

O R² dá cerca de 0,329. Para entender de onde vêm os limites dessa métrica, vale comparar com o modelo de referência mais simples possível: sempre prever a média de `y`, ignorando `x` completamente — o que corresponde a `alpha = mean(y)` e `beta = 0`. A soma dos erros ao quadrado desse modelo "vazio" é, por construção, exatamente igual a `total_sum_of_squares(y)`: sem informação nenhuma sobre `x`, o melhor palpite constante para cada ponto é a própria média. Isso dá R² = 0.

O modelo de mínimos quadrados precisa ser **pelo menos** tão bom quanto esse, porque ele é, por definição, a escolha de $\alpha$ e $\beta$ que minimiza a soma dos erros ao quadrado — e "sempre prever a média" é só mais uma escolha possível de $\alpha$ e $\beta$ entre todas as que ele considerou. Logo a soma dos erros ao quadrado do modelo ajustado é **no máximo** a soma dos erros ao quadrado do modelo vazio, o que trava o R² entre 0 e 1: quanto mais alto, melhor o ajuste.

> **⚠️ Atenção — 0,329 não é "bom" nem "ruim" — é o que os dados sustentam**
>
> Um R² de 0,329 diz que o número de amigos explica cerca de um terço da variação em quantos minutos alguém passa no site. Isso é um ajuste **razoável, mas nada além disso** — o modelo captura uma tendência real (a correlação de 0,57 já indicava isso), mas dois terços da variação vêm de outras coisas: hábitos individuais, o que cada pessoa faz na vida, fatores que este modelo de uma única variável não tem como enxergar. Um R² baixo não é sinal de erro de código; pode ser, e frequentemente é, sinal de que a variável escolhida não conta a história inteira.

O [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) deu o vocabulário para julgar modelos de classificação — acurácia, precisão, revocação. R² é a métrica equivalente para regressão: mede qualidade de ajuste, não corretude de categoria. A [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html), a seguir, dá a justificativa de fundo para minimizar exatamente essa soma de quadrados, em vez de qualquer outra medida de erro que também punisse desvios grandes.

> **💡 Dica — Na prática: `scikit-learn`**
>
> ```python
> from sklearn.linear_model import LinearRegression
>
> X = [[x] for x in num_friends_good]
> y = daily_minutes_good
>
> modelo = LinearRegression().fit(X, y)
> modelo.intercept_, modelo.coef_[0]   # alpha, beta
>
> modelo.score(X, y)                    # R²
> ```
>
> Vale rodar e comparar com o que as três funções desta seção devolveram, valor a valor:
>
> |  | à mão, nesta seção | `scikit-learn` 1.9 |
> |---|---|---|
> | $\alpha$ | `22.947552413468976` | `22.947552413468976` |
> | $\beta$ | `0.9038659456058725` | `0.9038659456058724` |
> | R² | `0.32910783778362984` | `0.32910783778362984` |
>
> $\alpha$ e o R² saem **idênticos**, bit a bit. O $\beta$ difere no último dígito da representação — um único bit de mantissa, a menor diferença que um `float64` sabe expressar, e nem sequer erro de método: é a ordem em que as somas de ponto flutuante foram feitas. As trinta e poucas linhas desta seção e a biblioteca que roda em produção no mundo inteiro estão calculando **a mesma coisa**.
>
> `LinearRegression` não itera e não pede taxa de aprendizado: ela resolve o sistema de mínimos quadrados por álgebra linear direta (decomposição em valores singulares), o equivalente industrial da fórmula fechada que você acabou de escrever à mão. `.score()` devolve o R² sem que você precise escrever `total_sum_of_squares` nem `r_squared` — mas é a mesma conta, e agora você sabe o que ela faz por dentro: soma de quadrados do modelo dividida pela soma de quadrados do modelo vazio, subtraída de 1.
>
> Os números da tabela foram medidos rodando o código, não estimados.

## Usando Gradiente Descendente

> **📌 Nota**
>
> Esta seção corresponde a *Using Gradient Descent*, do capítulo 14 de Grus (2019).

A [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) ajustou $\alpha$ e $\beta$ com uma fórmula fechada — duas médias, uma correlação, dois desvios padrão, sem laço nenhum. Isso só foi possível porque a regressão linear simples é um caso particularmente simples: derivar a soma dos erros ao quadrado à mão, igualar a zero e resolver o sistema dá conta do recado. Esta seção resolve o **mesmo problema**, com os **mesmos dados**, pelo caminho construído no [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html): gradiente descendente. O objetivo não é achar uma resposta melhor — é mostrar que os dois caminhos, o direto e o iterativo, chegam ao mesmo lugar.

Precisamos de três coisas da seção anterior: as funções `error` e `sum_of_sqerrors`, os dados da DataSciencester e o par $(\alpha, \beta)$ que a fórmula fechada já devolveu. Tudo isso vem de `scratch/simple_linear_regression.py` — é o mesmo código, só sem repetir:

In [ ]:
from scratch.simple_linear_regression import (
    error, sum_of_sqerrors,
    num_friends_good, daily_minutes_good,
    alpha as alpha_fechada, beta as beta_fechada,
)
import matplotlib.pyplot as plt
plt.close('all')

`alpha_fechada` e `beta_fechada` são os valores de mínimos quadrados que a seção anterior calculou com `least_squares_fit`. Eles são o **gabarito** desta seção: o número que o gradiente descendente vai ter que encontrar sozinho, sem nunca vê-lo.

In [ ]:
alpha_fechada, beta_fechada

### O gradiente da soma dos erros ao quadrado

Se escrevermos `theta = [alpha, beta]` — o vetor de parâmetros que a [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html) batizou assim, e que o código abaixo chama de `guess`, "o palpite" —, o problema é o mesmo que aquela seção já resolveu para uma reta com parâmetros conhecidos: minimizar o erro quadrático ajustando `theta` um passo de cada vez. O que muda é o dado — ali era uma reta perfeita, `y = 20x + 5`; aqui são os 203 pontos reais e ruidosos da DataSciencester.

Muda também a forma de escrever o gradiente, e a mudança não move o destino: as duas derivadas parciais aparecem soltas dentro do laço, em vez de embrulhadas numa função `linear_gradient`, e os gradientes são **somados** sobre os 203 pontos em vez de terem a média tirada. Somar em vez de tirar a média multiplica o gradiente inteiro por 203 — o que aponta exatamente para a mesma direção, só com comprimento outro, e por isso só pede uma taxa de aprendizado proporcionalmente menor. O fundo da tigela não se move.

Aqui está o laço inteiro. A perda de cada epoch vai para a barra de progresso e também para a lista `perdas` — são essas dez mil perdas que desenham a figura que fecha esta seção.

In [ ]:
import random
import tqdm
from scratch.gradient_descent import gradient_step

num_epochs = 10000
random.seed(0)

guess = [random.random(), random.random()]  # começa com um valor aleatório

learning_rate = 0.00001
perdas = []                                 # a perda a cada epoch

with tqdm.trange(num_epochs) as t:
    for _ in t:
        alpha, beta = guess

        # derivada parcial da perda em relação a alpha
        grad_a = sum(2 * error(alpha, beta, x_i, y_i)
                     for x_i, y_i in zip(num_friends_good,
                                         daily_minutes_good))

        # derivada parcial da perda em relação a beta
        grad_b = sum(2 * error(alpha, beta, x_i, y_i) * x_i
                     for x_i, y_i in zip(num_friends_good,
                                         daily_minutes_good))

        # perda, para mostrar na barra de progresso e para guardar
        loss = sum_of_sqerrors(alpha, beta,
                               num_friends_good, daily_minutes_good)
        perdas.append(loss)
        t.set_description(f"loss: {loss:.3f}")

        # finalmente, atualiza o palpite
        guess = gradient_step(guess, [grad_a, grad_b], -learning_rate)

alpha, beta = guess
assert 22.9 < alpha < 23.0
assert 0.9 < beta < 0.905

alpha, beta

> **📌 Nota**
>
> `grad_a` e `grad_b` são exatamente `linear_gradient` da [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html) escrita por extenso: a soma (não a média) das derivadas de `error(...) ** 2` sobre todos os pontos, uma parcial para cada parâmetro. `gradient_step` é a mesma função de cinco linhas que o Capítulo 5 construiu e que este livro reaproveita em todo capítulo que ajusta um modelo — aqui ela recebe a soma dos gradientes em vez da média, o que só reescala a taxa de aprendizado, não muda o destino.
>
> `learning_rate = 0.00001` é bem menor que o `0.001` usado na seção 5.5. Faz sentido: ali `x` variava de -50 a 49; aqui `x` é número de amigos, com a mesma ordem de grandeza, mas o gradiente soma sobre 203 pontos em vez de usar a média — uma soma maior pede um passo menor para não disparar para fora da tigela, a mesma lição da [seção 5.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/04-escolhendo-o-tamanho-do-passo.html).

### As duas respostas, lado a lado

In [ ]:
print(f"{'':<7}{'fórmula fechada':>20}{'gradiente descendente':>22}{'diferença':>12}")
for nome, fechada, iterativa in [("alpha", alpha_fechada, alpha),
                                 ("beta", beta_fechada, beta)]:
    print(f"{nome:<7}{fechada:>20.12f}{iterativa:>22.12f}"
          f"{abs(fechada - iterativa):>12.3g}")

Dez mil epochs de gradiente descendente, partindo de um `theta` sorteado ao acaso e sem nunca ver a fórmula fechada, chegaram ao mesmo $\alpha$ e ao mesmo $\beta$ que a álgebra da seção anterior devolveu de uma vez. E "o mesmo" aqui merece um número, não um advérbio: as duas respostas diferem em **2,6 × 10⁻⁷** no intercepto e **2,1 × 10⁻⁸** na inclinação. A discordância só aparece na sétima casa decimal de um intercepto medido em dezenas de minutos, e na oitava de uma inclinação medida em minutos por amigo — bem além de qualquer precisão que os dados de origem tenham. Para efeito prático, os dois métodos devolveram o mesmo par de números.

In [ ]:
# Figura: Soma dos erros ao quadrado a cada epoch do gradiente descendente, contra a perda da solução de fórmula fechada (escala log no eixo y)
perda_fechada = sum_of_sqerrors(alpha_fechada, beta_fechada,
                                num_friends_good, daily_minutes_good)

def pt_br(x: float) -> str:
    """13196.6 -> '13.196,6' — troca o ponto e a vírgula de lugar"""
    return f"{x:,.1f}".translate(str.maketrans(",.", ".,"))

plt.plot(range(1, num_epochs + 1), perdas, label="gradiente descendente")
plt.axhline(perda_fechada, color="black", linestyle="--",
            label=f"fórmula fechada ({pt_br(perda_fechada)})")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("soma dos erros ao quadrado")
plt.legend()
plt.show()

A figura mostra o caminho que a tabela esconde. A perda começa perto de $1{,}2 \times 10^5$: `theta` nasceu como dois números sorteados entre 0 e 1, e uma reta dessas erra feio nos 203 pontos. Os dez primeiros epochs sozinhos já a derrubam para menos de $6 \times 10^4$ — é o trecho quase vertical na borda esquerda. Daí em diante a queda desacelera, e pouco depois do epoch 1.500 a curva encosta na tracejada da solução exata; por volta do 2.000 as duas ficam indistinguíveis, e assim seguem até o fim.

Indistinguível não é igual. No epoch 2.000, a perda ainda está 29,8 acima da perda da fórmula fechada — 0,23% dela, menos de um pixel nesta escala. No 5.000, a distância é de $5 \times 10^{-4}$; no último epoch, de $5 \times 10^{-12}$. Os oito mil epochs do trecho reto à direita não estão parados: estão limpando doze ordens de grandeza que nenhum eixo poderia mostrar junto com a queda inicial. Repare, por fim, que a curva se aproxima da tracejada **sempre por cima**, sem nunca cruzá-la. Não é sorte do desenho: a fórmula fechada devolve o mínimo da função, e não existe `theta` com perda menor para o gradiente descendente encontrar.

> **🔷 Conceito**
>
> Isso é a promessa da [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html) cumprida por inteiro, e vale dizer o argumento inteiro de uma vez, porque ele estava partido entre as duas seções.
>
> Minimizar a soma dos erros ao quadrado significa achar o ponto em que o gradiente dela se anula. A **fórmula fechada** da seção 11.1 resolve essa condição **algebricamente**: derive `sum_of_sqerrors` em relação a $\alpha$ e a $\beta$, iguale as duas derivadas a zero, isole — e o que sai são médias, variância e covariância. O **gradiente descendente** desta seção procura o mesmo ponto **numericamente**: avalia o gradiente onde está, dá um passo contra ele e repete — dez mil vezes, aqui —, até que os passos deixem de mudar `theta` porque o gradiente já é praticamente nulo. É a **mesma equação**, resolvida de dois jeitos — um que a escreve e a isola, outro que a persegue.
>
> Falta uma peça, e é a que a convexidade fornece. A seção 5.5 mostrou que a soma dos erros ao quadrado de um modelo linear é uma função **convexa** de `theta` — uma tigela de verdade, sem vales escondidos. Convexa significa que existe **um só** ponto onde o gradiente se anula. Sem isso, "resolver a equação do gradiente nulo" poderia ter várias respostas, e o método iterativo poderia escorregar para um fundo secundário que a álgebra não escolheu — os dois métodos concordariam por sorte, ou não concordariam. É a unicidade do ponto que transforma a coincidência numérica acima em consequência necessária: existe um fundo, é único, e os dois caminhos só podem terminar nele.

Vale perguntar por que passar pelo caminho mais lento, já que a fórmula fechada existe e é exata. A resposta é que ela **só** existe porque este problema é simples o bastante: uma única variável explicativa e uma função de erro cuja derivada dá para igualar a zero e resolver à mão. A [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html), a seguir, explica de onde vem essa função de erro específica. E assim que o problema cresce um pouco — mais de uma variável explicativa, no [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) — a álgebra ainda fecha, mas fica pesada o bastante para valer a pena revisitar. A regressão logística do [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html) e as redes neurais dos capítulos seguintes não têm fórmula fechada **nenhuma**: para elas, o gradiente descendente que você acabou de usar aqui não é uma alternativa mais lenta — é o único caminho que existe.

> **💡 Dica — Na prática: diferenciação automática**
>
> `grad_a` e `grad_b` acima foram derivadas à mão: alguém (o Grus, neste caso) pegou `sum_of_sqerrors`, aplicou a regra da cadeia em relação a cada parâmetro, e transcreveu o resultado em código. Para dois parâmetros isso é tedioso, mas viável. Para uma rede neural com milhares ou milhões de parâmetros — assunto dos capítulos [15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) — é inviável.
>
> Bibliotecas como PyTorch e TensorFlow resolvem isso com **diferenciação automática** (*autodiff*): em vez de pedir a derivada pronta, elas registram cada operação aplicada aos parâmetros — multiplicação, soma, potência — e aplicam a regra da cadeia automaticamente, operação por operação, na volta. O nome técnico do caso que aparece em redes neurais é *backpropagation*, e é literalmente a mesma ideia que `grad_a` e `grad_b` calculam aqui, só que mecanizada para não depender de alguém derivar a função à mão a cada modelo novo.
>
> O gradiente descendente em si — dar um passo pequeno na direção oposta ao gradiente, repetir — não muda nada. O que a diferenciação automática elimina é exatamente o trabalho manual que este chunk fez.

## Estimação por Máxima Verossimilhança

> **📌 Nota**
>
> Esta seção corresponde a *Maximum Likelihood Estimation*, do capítulo 14 de Grus (2019).

As duas seções anteriores minimizaram a mesma quantidade por dois caminhos diferentes: a soma dos erros ao quadrado. Mas por que **essa** soma? Somar os erros absolutos também puniria desvios grandes. Somar os erros elevados à quarta potência também. Cada uma dessas escolhas produz um modelo diferente, e até aqui a única razão dada para preferir o quadrado foi que ele tem fórmula fechada (seção 11.1) e forma de tigela convexa, fácil de descer (seção 11.2). Conveniência matemática não é a mesma coisa que estar certo.

Esta seção dá a justificativa que faltava: **estimação por máxima verossimilhança**. Ela mostra que, sob uma suposição específica e razoável sobre os erros, minimizar a soma dos erros ao quadrado não é uma escolha entre outras — é a **única** escolha consistente com essa suposição.

### O que é verossimilhança

Comece de um jeito mais geral, sem regressão ainda. Imagine uma amostra de dados $v_1, \ldots, v_n$ vinda de uma distribuição que depende de um parâmetro desconhecido $\theta$ (theta). Atenção ao nome: nas seções 11.2 e 5.5, `theta` era o vetor de parâmetros **do modelo** — inclinação e intercepto juntos; aqui, por seguir a notação usual de estatística, $\theta$ é o parâmetro desconhecido **da distribuição** de onde os dados vieram. É o mesmo papel — "aquilo que não se sabe e se quer estimar" — em dois contextos diferentes, e daqui a duas subseções os dois se encontram: o $\theta$ desta seção vai ser exatamente o par $(\alpha, \beta)$.

A probabilidade de observar exatamente essa amostra, dado $\theta$, é:

$$
p(v_1, \ldots, v_n \mid \theta)
$$

Essa expressão responde à pergunta "conhecido $\theta$, quão provável é ver estes dados?". A ideia central da máxima verossimilhança é girar a pergunta ao contrário: **os dados já foram vistos** — são fixos, estão na sua tela. O que não se sabe é $\theta$. Encarada assim, a mesma expressão vira uma função de $\theta$, chamada de **verossimilhança**:

$$
L(\theta \mid v_1, \ldots, v_n)
$$

> **🔷 Conceito**
>
> Um valor de $\theta$ que tornaria os dados observados extremamente improváveis é, intuitivamente, um mau candidato — se aquele $\theta$ fosse o real, seria uma coincidência e tanto ver justamente estes dados. O $\theta$ **mais plausível**, sob essa lógica, é o que torna os dados que você de fato observou os **menos surpreendentes possível**: o valor de $\theta$ que maximiza $L(\theta \mid v_1, \ldots, v_n)$.
>
> Isso vale tanto para distribuições discretas (onde $p$ é uma massa de probabilidade) quanto para distribuições contínuas (onde $p$ é uma densidade) — a lógica de virar a pergunta ao contrário não muda.

### Aplicando à regressão

Uma suposição comum sobre o modelo de regressão — e é uma suposição, feita de fora, não algo que os dados provem sozinhos — é que os erros $\varepsilon_i$ são normalmente distribuídos, com média 0 e um desvio padrão $\sigma$ **fixo, o mesmo para todos os pontos** — a dispersão em torno da reta não pode crescer conforme `x` cresce. (Essa segunda exigência tem nome: **homocedasticidade**. O [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) volta a ela, porque é uma das suposições que se checa antes de confiar num erro padrão de coeficiente.) Não é uma suposição arbitrária: quando o erro é, de fato, a soma de muitos fatores pequenos e independentes — o humor do usuário naquele dia, o que mais estava competindo por atenção dele, um problema de rede — esse tipo de soma tende a se comportar como uma normal, mesmo sem nenhum dos fatores individuais ser normal. Mas continua sendo uma suposição, não uma consequência automática dos dados, e vale checá-la quando o modelo importa de verdade.

Sob essa suposição, a verossimilhança de ver um único par $(x_i, y_i)$, dados candidatos $\alpha$ e $\beta$, é a densidade normal avaliada no erro daquele ponto:

$$
L(\alpha, \beta \mid x_i, y_i, \sigma) = \frac{1}{\sqrt{2\pi}\,\sigma} \exp\left(-\frac{(y_i - \alpha - \beta x_i)^2}{2\sigma^2}\right)
$$

Repare que $y_i - \alpha - \beta x_i$ é exatamente o negativo do que a função `error` da [seção 11.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) calcula — e como ele aparece elevado ao quadrado, o sinal não importa.

> **🔷 Conceito**
>
> Vale parar na forma dessa expressão antes de seguir para a álgebra, porque ela já contém o argumento inteiro.
>
> O termo dentro do `exp` é **negativo** e **quadrático** no erro. Um candidato $(\alpha, \beta)$ que erra bastante para algum ponto $x_i$ — produz um resíduo grande — faz esse termo despencar para um número muito negativo, e `exp` de um número muito negativo é um fator perto de zero: aquele ponto se torna "surpreendente" sob esse candidato, e puxa a verossimilhança inteira para baixo.
>
> E porque o erro está **ao quadrado** dentro do `exp`, dobrar o tamanho de um resíduo não dobra a penalidade — ela cresce muito mais rápido que isso, porque dobrar o resíduo quadruplica o expoente. Um modelo que erra um pouco em todo lugar é julgado com muito mais indulgência, sob esta conta, do que um modelo que acerta quase tudo e erra feio num único ponto. É a mesma preferência — por muitos erros pequenos em vez de um erro enorme — que já vimos a soma dos erros ao quadrado impor nas duas seções anteriores. Não é coincidência: é a mesma conta, vista de dois ângulos.

### Por que isso equivale a mínimos quadrados

Os erros de pontos diferentes são tratados como independentes entre si, então a verossimilhança do conjunto de dados inteiro é o **produto** das verossimilhanças individuais:

$$
L(\alpha, \beta \mid \text{dados}, \sigma) = \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi}\,\sigma} \exp\left(-\frac{(y_i - \alpha - \beta x_i)^2}{2\sigma^2}\right)
$$

O produto de exponenciais é a exponencial da soma dos expoentes, e o fator $\frac{1}{\sqrt{2\pi}\,\sigma}$ não depende de $\alpha$ nem de $\beta$ — só se repete $n$ vezes. Isolando o que de fato varia com $\alpha$ e $\beta$:

$$
L(\alpha, \beta \mid \text{dados}, \sigma) = \left(\frac{1}{\sqrt{2\pi}\,\sigma}\right)^{n} \exp\left(-\frac{1}{2\sigma^2} \sum_{i=1}^{n} (y_i - \alpha - \beta x_i)^2\right)
$$

E o somatório dentro do `exp` é, a menos do sinal já discutido, exatamente `sum_of_sqerrors(alpha, beta, x, y)` — a mesma função que a [seção 11.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) escreveu. Chame essa soma de $S(\alpha, \beta)$. A expressão inteira fica $\left(\frac{1}{\sqrt{2\pi}\sigma}\right)^n \exp\left(-\frac{S(\alpha, \beta)}{2\sigma^2}\right)$ — uma constante positiva (que não depende de $\alpha, \beta$) vezes $\exp(-S(\alpha, \beta) / 2\sigma^2)$.

`exp` é uma função estritamente crescente, e $1/2\sigma^2$ é um número positivo fixo (não depende de $\alpha$ nem $\beta$). Maximizar $\exp(-S(\alpha,\beta) / 2\sigma^2)$ sobre $(\alpha, \beta)$ é, portanto, exatamente o mesmo problema que **minimizar** $S(\alpha, \beta)$ sobre $(\alpha, \beta)$.

> **❗ Importante — O argumento inteiro, numa frase**
>
> Sob a suposição de que os erros são normais, o $(\alpha, \beta)$ que **maximiza a verossimilhança dos dados observados** é exatamente o mesmo $(\alpha, \beta)$ que **minimiza a soma dos erros ao quadrado**. Mínimos quadrados não é uma escolha de conveniência: é a consequência direta de assumir erros normais. Escolher outra função de perda — erro absoluto, por exemplo — corresponde, por este mesmo raciocínio ao contrário, a assumir uma distribuição diferente para o erro (para o erro absoluto, uma distribuição de Laplace).

### Vendo isso nos dados

O argumento acima é sobre a forma da função, não sobre nenhum $\sigma$ específico — o resultado vale para qualquer $\sigma$ fixo, porque $\sigma$ nunca aparece na comparação entre candidatos. Ainda assim, ajuda ver a diferença de escala em números concretos. A [seção 11.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) já comparou o modelo de mínimos quadrados com o modelo de referência mais simples possível — sempre prever a média de `y`, ignorando `x` — para calcular R². Vamos reaproveitar essa comparação aqui:

In [ ]:
from scratch.simple_linear_regression import (
    error, sum_of_sqerrors, num_friends_good, daily_minutes_good,
    alpha, beta,
)
from scratch.statistics import mean
import math
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
S_minimos_quadrados = sum_of_sqerrors(alpha, beta,
                                      num_friends_good, daily_minutes_good)
S_sempre_a_media = sum_of_sqerrors(mean(daily_minutes_good), 0,
                                   num_friends_good, daily_minutes_good)

S_minimos_quadrados, S_sempre_a_media

O modelo de mínimos quadrados soma cerca de 13.197 de erro ao quadrado; sempre prever a média soma cerca de 19.670 — a mesma variação total que a seção 11.1 chamou de `total_sum_of_squares`. Qualquer que seja $\sigma$, o fator $\exp(-S/2\sigma^2)$ do modelo de mínimos quadrados é maior do que o do modelo "sempre a média", porque seu expoente é menos negativo: $S$ é bem menor. Sob a suposição de erros normais, os dados observados são muito mais prováveis com o $(\alpha, \beta)$ que a seção 11.1 encontrou do que com qualquer alternativa que ignore `x` — inclusive a mais simples de todas.

### Por que ninguém calcula a verossimilhança direto

A álgebra acima passou do produto para a exponencial da soma em uma linha, como se as duas expressões fossem intercambiáveis. Elas são — no papel. Num computador, não.

Escreva as duas versões e compare. A primeira multiplica as 203 densidades normais, uma por ponto, exatamente como a fórmula do produto manda. A segunda usa $\log(ab) = \log a + \log b$ para trocar o produto por uma soma de logaritmos, e é a **log-verossimilhança**. Repare que ela não calcula a densidade para depois tirar o log: como o log de uma exponencial é o próprio expoente, o termo de cada ponto sai direto como $-e_i^2/2\sigma^2 - \log(\sqrt{2\pi}\,\sigma)$. O número minúsculo nunca chega a existir na memória.

In [ ]:
def verossimilhanca(alpha: float, beta: float, x, y, sigma: float) -> float:
    """L(alpha, beta | dados, sigma): o produto das densidades normais."""
    produto = 1.0
    for x_i, y_i in zip(x, y):
        e = error(alpha, beta, x_i, y_i)
        produto *= (math.exp(-e ** 2 / (2 * sigma ** 2))
                    / (math.sqrt(2 * math.pi) * sigma))
    return produto

def log_verossimilhanca(alpha: float, beta: float, x, y, sigma: float) -> float:
    """log L: a mesma conta, com a soma dos logs no lugar do produto."""
    return sum(-error(alpha, beta, x_i, y_i) ** 2 / (2 * sigma ** 2)
               - math.log(math.sqrt(2 * math.pi) * sigma)
               for x_i, y_i in zip(x, y))

Agora avalie as duas nos mesmos dois modelos que acabamos de comparar. Qualquer $\sigma$ fixo serve para o argumento; usamos 8, que é aproximadamente o tamanho do resíduo típico deste ajuste:

In [ ]:
sigma = 8.0
media = mean(daily_minutes_good)

L_ajustado = verossimilhanca(alpha, beta,
                             num_friends_good, daily_minutes_good, sigma)
L_media = verossimilhanca(media, 0,
                          num_friends_good, daily_minutes_good, sigma)

print(f"L  (mínimos quadrados) = {L_ajustado!r}")
print(f"L  (sempre a média)    = {L_media!r}")

In [ ]:
logL_ajustado = log_verossimilhanca(alpha, beta,
                                    num_friends_good, daily_minutes_good, sigma)
logL_media = log_verossimilhanca(media, 0,
                                 num_friends_good, daily_minutes_good, sigma)

print(f"log L (mínimos quadrados) = {logL_ajustado:.4f}")
print(f"log L (sempre a média)    = {logL_media:.4f}")
print(f"diferença                 = {logL_ajustado - logL_media:.4f}")

> **⚠️ Atenção — O produto certo é o número errado**
>
> Olhe o que o produto direto devolveu. A verossimilhança do modelo de mínimos quadrados saiu na casa de $10^{-310}$ — abaixo de $2{,}2 \times 10^{-308}$, o menor `float64` **normal**. Ela ainda é representável, mas só como *subnormal*: uma faixa em que o ponto flutuante compra alcance vendendo precisão, e cada divisão por dois custa um bit de mantissa que não volta.
>
> E a verossimilhança do modelo "sempre a média" saiu **exatamente `0.0`**. Não é que ela seja zero — a fórmula é um produto de densidades, todas positivas, e a log-verossimilhança logo abaixo dá o valor: $e^{-762{,}345}$, isto é, algo perto de $10^{-331}$. Simplesmente não há expoente em `float64` que guarde isso, e o produto despencou para zero levando junto toda a informação. Com esse zero na mão, a comparação entre os dois modelos se degrada: `L_ajustado > L_media` ainda é verdade, mas só porque o outro escapou do zero por pouco; `L_media / L_ajustado` dá 0, e qualquer terceiro candidato pior também zeraria e ficaria empatado com este.
>
> A log-verossimilhança não tem esse problema. Os mesmos dois modelos dão −711,77 e −762,35: números perfeitamente confortáveis, com a diferença de 50,58 intacta e legível. Como `exp` é crescente, comparar log-verossimilhanças ordena os modelos exatamente como comparar verossimilhanças ordenaria — só que a conta existe.
>
> É a mesma armadilha, e a mesma saída, da [seção sobre *underflow* do Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/02-um-filtro-mais-sofisticado.html): lá eram milhares de probabilidades por palavra, aqui são 203 densidades normais. É por isso que a literatura fala em **maximizar a log-verossimilhança**, praticamente nunca em maximizar a verossimilhança: não é preferência de notação, é a única das duas que um computador consegue calcular.

### O que isso amarra

Este capítulo montou o mesmo resultado três vezes, cada vez respondendo a uma pergunta diferente:

- A [seção 11.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) respondeu **como** ajustar $\alpha$ e $\beta$ diretamente, com uma fórmula fechada que só existe porque este problema é simples o bastante para resolver a álgebra à mão.
- A [seção 11.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/02-usando-gradiente-descendente.html) respondeu **como** ajustar os mesmos parâmetros iterando, pelo caminho geral do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) — o caminho que continua funcionando mesmo quando a fórmula fechada desaparece.
- Esta seção respondeu **por que** minimizar a soma dos erros ao quadrado, para começo de conversa, era a coisa certa a fazer.

A suposição de erros normais é o que amarra as três. O próximo capítulo mantém a suposição inteira e mexe em outro lugar: o [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html), sobre regressão múltipla, continua supondo erros normais de média zero e $\sigma$ fixo, e continua minimizando soma de quadrados — só troca uma variável explicativa por várias, e com isso ganha um problema novo, o de decidir quais variáveis merecem estar no modelo. Tudo o que esta seção justificou continua valendo lá, palavra por palavra.

A suposição só cai um capítulo depois. Quando `y` deixa de ser uma quantidade contínua e vira uma decisão binária (sim ou não, comprou ou não comprou), o raciocínio de máxima verossimilhança continua de pé, mas a distribuição assumida para os dados muda, de normal para Bernoulli. É exatamente o que o [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html), sobre regressão logística, faz: a verossimilhança vira outra função, a perda deixa de ser soma de quadrados e passa a ser log-verossimilhança negativa — o mesmo espaço de log que acabamos de usar para escapar do *underflow* —, e, porque essa nova função não tem uma álgebra tão gentil quanto esta, a fórmula fechada desaparece de vez. O caminho da seção 11.2 é o único que sobra.

> **💡 Dica — Na prática: o que se faz com isso**
>
> Nenhuma chamada de `scikit-learn` pede para você escrever uma verossimilhança. `LinearRegression().fit(X, y)` resolve o sistema de mínimos quadrados sem perguntar nada sobre a distribuição dos erros — o argumento desta seção já está embutido na escolha do método, não no código que você chama.
>
> Mas a suposição de erros normais volta à tona no momento em que se quer mais do que uma previsão pontual: intervalos de confiança para $\alpha$ e $\beta$, testes de hipótese sobre se $\beta$ é realmente diferente de zero, ou um valor-p. Ferramentas como `statsmodels`, em Python, calculam tudo isso — e todos esses números dependem, na base, da mesma suposição de normalidade que esta seção acabou de justificar. O [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html), sobre regressão múltipla, chega perto desse território ao discutir erros padrão dos coeficientes: a pergunta "esse coeficiente é confiável, ou pode ser só ruído?" é, no fundo, a mesma pergunta desta seção, aplicada de novo.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 14 de Grus (2019) tem uma única sugestão: continuar lendo sobre regressão múltipla. É exatamente o que o [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) deste livro faz — o mesmo modelo deste capítulo, com mais de uma variável explicativa de cada vez. Lá a álgebra ainda fecha (a solução de mínimos quadrados de uma regressão múltipla também tem forma exata), mas fica pesada o bastante para que o caminho iterativo da seção 11.2 passe a ser o mais prático — e é por ele que o Capítulo 12 vai.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.